# Midstream result 2023

In [2]:
import pandas as pd

In [3]:
# contribution data
path = "/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/leontief/"

prod_contribution = pd.read_csv(path + "row_contribution_percentage_production_long_gt1_scaled100.csv")
rnd_contribution = pd.read_csv(path + "column_contribution_percentage_receiving_long_gt1_scaled100.csv")

In [4]:
# production data
path = '/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/state production consumption/state_production_2023_with_ids.csv'
production_data = pd.read_csv(path)

In [ ]:
import pandas as pd

path_data = '/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/state production consumption/'
path_leon = "/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/leontief/"
# File paths
row_contrib_fp = path_leon+"row_contribution_percentage_production_long_gt1_scaled100.csv"
state_prod_fp = path_data+"state_production_2023_with_ids.csv"

# Read files
row_contrib_df = pd.read_csv(row_contrib_fp)
state_prod_df = pd.read_csv(state_prod_fp)

# Remove rows where source == 'US'
row_contrib_df = row_contrib_df[row_contrib_df['source'] != 'US']

# Merge: 'source' in row_contrib_df to 'state' in state_prod_df
merged_df = pd.merge(row_contrib_df, state_prod_df, left_on='source', right_on='state', how='left')

# Calculate transported gas
merged_df['transported_gas'] = (
    merged_df['natural_gas_dry_production_MMcf'] *
    merged_df['contribution_percentage'] * 0.01
)

# Save result
merged_df.to_csv(path_leon + "merged_transported_gas.csv", index=False)

print("Saved merged_transported_gas_from_production.csv")


Saved merged_transported_gas.csv


In [8]:

# File paths
col_contrib_fp = path_leon+"column_contribution_percentage_receiving_long_gt1_scaled100.csv"
state_cons_fp = path_data+"state_consumption_2023_with_ids.csv"

# Read files
col_contrib_df = pd.read_csv(col_contrib_fp)
state_cons_df = pd.read_csv(state_cons_fp)

# Remove rows where source == 'US'
col_contrib_df = col_contrib_df[col_contrib_df['source'] != 'US']

# Merge: 'destination' in col_contrib_df to 'state' in state_cons_df
merged_df = pd.merge(col_contrib_df, state_cons_df, left_on='destination', right_on='state', how='left')

# Calculate transported gas
merged_df['transported_gas'] = (
    merged_df['natural_gas_consumption_value_MMcf'] *
    merged_df['contribution_percentage'] * 0.01
)

# Save result
merged_df.to_csv(path_leon+"merged_transported_gas_consumption.csv", index=False)

print("Saved merged_transported_gas_consumption.csv")


Saved merged_transported_gas_consumption.csv


In [ ]:
# deals with missing distance
path_mid = "/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/midstream_state_level/"
dist_df  = pd.read_csv(path_mid+ "processing_to_delivery_distance_long_km.csv")     # processing_id, delivery_id, distance_km
flows_df = pd.read_csv(path_mid+ "merged_transported_gas_consumption.csv")          # source, destination, ...

merged = flows_df.merge(
    dist_df[['processing_id', 'delivery_id', 'distance_km']],
    left_on=['source', 'destination'],
    right_on=['processing_id', 'delivery_id'],
    how='left'  # keep all flows; attach distance when available
).drop(columns=['processing_id', 'delivery_id'])

# Identify labels
texas_label = "Texas"
louisiana_label = "Louisiana"

# Mean distance from TX & LA to each destination
mean_dist_by_dest = (
    dist_df[dist_df["processing_id"].isin([texas_label, louisiana_label])]
    .groupby("delivery_id")["distance_km"].mean()
)

# Apply only to rows where source is Federal Offshore--Gulf of America
mask = merged["source"] == "Federal Offshore--Gulf of America"
merged.loc[mask, "distance_km"] = merged.loc[mask, "destination"].map(mean_dist_by_dest)

# 75th percentile of all distances (km)
p75_km = dist_df["distance_km"].quantile(0.75)

# case-insensitive match for "International"
mask = merged["source"].astype(str).str.strip().str.lower().eq("international") | \
       merged["destination"].astype(str).str.strip().str.lower().eq("international")

merged.loc[mask, "distance_km"] = p75_km

merged.to_csv(path_mid+"flows_with_distances.csv", index=False)


In [15]:
path_mid = "/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/midstream_state_level/"
midstream_df = pd.read_csv(path_mid + "flows_with_distances_filled.csv")

In [17]:
# Constants (match your R code)
midstream_emission_factor = 4.00        # kgCO2e/MMCF-km
midstream_emission_factor_low = 3.77
midstream_emission_factor_high = 4.28
tortuosity_factor_low = 1.03
tortuosity_factor_high = 1.16
tortuosity_factor = (tortuosity_factor_low + tortuosity_factor_high) / 2.0
MJ_per_MMCF = 1094000                # MJ/MMCF
kgCO2_mmcf_to_g_MJ = 1000.0 / MJ_per_MMCF  # kg/mmscf -> g/MJ

# Read input
df = pd.read_csv(path_mid+"flows_with_distances_filled.csv")

# Compute EFs (gCO2e/MJ)
km = df['distance_km'] 
df['midstream_EF_g_MJ'] = km * midstream_emission_factor * kgCO2_mmcf_to_g_MJ * tortuosity_factor
df['midstream_EF_g_MJ_low'] = km * midstream_emission_factor_low * kgCO2_mmcf_to_g_MJ * tortuosity_factor_low
df['midstream_EF_g_MJ_high'] = km * midstream_emission_factor_high * kgCO2_mmcf_to_g_MJ * tortuosity_factor_high

# Save
df.to_csv(path_mid+"flows_with_midstream_ef.csv", index=False)


In [35]:
path_data = '/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/state production consumption/'
prod_path = path_data+"state_production_2023_with_ids.csv"
cons_path = path_data+"state_consumption_2023_with_ids.csv"

prod_df = pd.read_csv(prod_path)
cons_df = pd.read_csv(cons_path)

prod_col = 'natural_gas_dry_production_MMcf'
cons_col = 'natural_gas_consumption_value_MMcf'

# Make sure the column exists and is numeric
prod_df[prod_col] = pd.to_numeric(prod_df[prod_col], errors='coerce')
prod_df = prod_df[prod_df['state'] != "U.S."]
prod_df = prod_df[prod_df['state'] != "International"]
# If there can be multiple rows per state, aggregate first
prod_by_state = (
    prod_df.groupby('state', as_index=False)[prod_col]
           .sum()
)
# Sort and take top 10
top10_prod = prod_by_state.sort_values(prod_col, ascending=False).head(10)
# (Optional) get just the list of state names, in order
top10_prod = top10_prod['state'].tolist()

# Make sure the column exists and is numeric
cons_df[cons_col] = pd.to_numeric(cons_df[cons_col], errors='coerce')
cons_df = cons_df[cons_df['state'] != "U.S."]
cons_df = cons_df[cons_df['state'] != "International"]
# If there can be multiple rows per state, aggregate first
cons_by_state = (
    cons_df.groupby('state', as_index=False)[cons_col]
           .sum()
)
# Sort and take top 10
top10_cons = cons_by_state.sort_values(cons_col, ascending=False).head(10)
# (Optional) get just the list of state names, in order
top10_cons = top10_cons['state'].tolist()

print(top10_prod)
print(top10_cons)

['Texas', 'Pennsylvania', 'Louisiana', 'West Virginia', 'Oklahoma', 'New Mexico', 'Ohio', 'Colorado', 'Wyoming', 'North Dakota']
['Texas', 'California', 'Louisiana', 'Pennsylvania', 'Florida', 'New York', 'Ohio', 'Illinois', 'Michigan', 'Indiana']


In [80]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# --- Load & filter (adjust as you like) ---
df = pd.read_csv(path_mid + "flows_with_midstream_ef.csv")
df = df[df['source'].isin(top10_prod) & df['destination'].isin(top10_cons)]
df = df[~((df['source'] == 'International') & (df['destination'] == 'International'))]

#df = df.nlargest(30, "transported_gas").copy()   # or 10, or all

# Labels for two distinct node sets
df["source_label"] = df["source"].astype(str) + " prod"
df["dest_label"]   = df["destination"].astype(str) + " cons"

# Order nodes by total volume (nice layout)
prod_order = (df.groupby("source_label")["transported_gas"]
                .sum().sort_values(ascending=False).index.tolist())
cons_order = (df.groupby("dest_label")["transported_gas"]
                .sum().sort_values(ascending=False).index.tolist())

nodes = prod_order + cons_order
node_id = {name: i for i, name in enumerate(nodes)}

# Fix positions: prod at x=0, cons at x=1 (keeps two distinct columns)
x_prod = [0.1] * len(prod_order)
x_cons = [1.0] * len(cons_order)
y_prod = np.linspace(0.02, 0.98, len(prod_order))  # spread vertically
y_cons = np.linspace(0.02, 0.98, len(cons_order))
node_x = x_prod + x_cons
node_y = y_prod.tolist() + y_cons.tolist()

# Color links by midstream CI
norm  = mcolors.Normalize(vmin=df["midstream_EF_g_MJ"].min(),
                          vmax=df["midstream_EF_g_MJ"].max())
cmap  = cm.get_cmap("RdYlGn_r")  # high CI = red, low CI = green
link_colors = [mcolors.to_hex(cmap(norm(ci))) for ci in df["midstream_EF_g_MJ"]]

import re

# ... your existing code up to nodes/node_id ...

# Labels shown on the graph (strip " prod"/" cons" for display only)
nodes_display = [re.sub(r"\s+(prod|cons)\b", "", n) for n in nodes]

fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=14,
        thickness=16,
        label=nodes_display,    # <- display without suffixes
        color="lightgrey",
        x=node_x,
        y=node_y
    ),
    link=dict(
        source=df["source_label"].map(node_id),
        target=df["dest_label"].map(node_id),
        value=df["transported_gas"],
        color=link_colors,
        hovertemplate=(
            "From %{source.label} → %{target.label}<br>" +
            "Transported gas: %{value:,}<br>" +
            "Midstream CI (g/MJ): %{customdata:.3f}<extra></extra>"
        ),
        customdata=df["midstream_EF_g_MJ"]
    )
))


fig.update_layout(
    font=dict(family="Helvetica, Arial, sans-serif", size=12),
    hoverlabel=dict(font_family="Helvetica, Arial, sans-serif")
)

# # ---------- 1) Color legend (continuous colorbar for midstream CI) ----------
ci_min = float(df["midstream_EF_g_MJ"].min())
ci_max = float(df["midstream_EF_g_MJ"].max())

fig.add_trace(
    go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(
            colorscale="RdYlGn_r",
            cmin=ci_min, cmax=ci_max,
            color=[ci_min, ci_max],
            size=1,
            showscale=True,
            colorbar=dict(
                title="Midstream CI (g CO₂e/MJ)",
                thickness=26,            # <- width of the colorbar (px)
                thicknessmode="pixels",  # explicit; default is pixels
                len=0.8, lenmode="fraction",
                y=0.5, x=1.03,           # position
                bgcolor="white",
                outlinewidth=0,
                ticks="outside",
                tickfont=dict(size=11),
                titlefont=dict(size=12)
            )
        ),
        hoverinfo="none", showlegend=False
    )
)

fig.update_layout(
    template="plotly_white",        # or "none"
    paper_bgcolor="white",
    plot_bgcolor="white"
)

# Hide any axes created by the dummy scatter (remove grid, ticks, and axis lines)
fig.update_xaxes(visible=False, showgrid=False, zeroline=False, showline=False)
fig.update_yaxes(visible=False, showgrid=False, zeroline=False, showline=False)

# (Optional) tighten margins so legends fit nicely
fig.update_layout(margin=dict(l=10, r=100, t=60, b=10))

import re

# 1) Hide built-in node labels
fig.data[0].node.label = [""] * len(nodes)



fig.show()

# Vector export (crisp at any zoom)
fig.write_image("production_to_consumption_sankey_no_anno.svg", format="svg", width=800, height=400)


# volume weighted average midstream CI

In [82]:
# ---- Config ----
path = "/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/midstream_state_level/"
csv_path = path + "flows_with_midstream_ef.csv"  # change to your path if needed
weight_col = "transported_gas"
value_cols: list[str] = [
    "distance_km",
    "midstream_EF_g_MJ",
    "midstream_EF_g_MJ_low",
    "midstream_EF_g_MJ_high",
]

# ---- Load ----
df = pd.read_csv(csv_path)

# Ensure numeric types (coerce errors to NaN)
df[weight_col] = pd.to_numeric(df[weight_col], errors="coerce")
for c in value_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

def weighted_avg(g: pd.DataFrame, col: str, wcol: str = weight_col) -> float:
    """Volume-weighted average of `col` using `wcol` as weights, NaN-safe."""
    v = g[col]
    w = g[wcol]
    m = v.notna() & w.notna()
    if not m.any():
        return float("nan")
    w_sum = w[m].sum()
    if w_sum == 0:
        return float("nan")
    return float((v[m] * w[m]).sum() / w_sum)

def summarize(group_key: str) -> pd.DataFrame:
    """Group by group_key and return weighted averages + total transported_gas."""
    grp = df.groupby(group_key, dropna=False)
    out = grp.apply(
        lambda g: pd.Series(
            {
                "distance_km_weighted":            weighted_avg(g, "distance_km"),
                "midstream_EF_g_MJ_weighted":      weighted_avg(g, "midstream_EF_g_MJ"),
                "midstream_EF_g_MJ_low_weighted":  weighted_avg(g, "midstream_EF_g_MJ_low"),
                "midstream_EF_g_MJ_high_weighted": weighted_avg(g, "midstream_EF_g_MJ_high"),
                "total_transported_gas":           g[weight_col].sum(skipna=True),
            }
        )
    ).reset_index()
    return out

# ---- Results ----
by_source = summarize("source")
by_destination = summarize("destination")

# Optional: sort by total transported gas (descending)
by_source = by_source.sort_values("total_transported_gas", ascending=False)
by_destination = by_destination.sort_values("total_transported_gas", ascending=False)

# Optional: save
by_source.to_csv(path + "midstream_CI_weighted_by_source.csv", index=False)
by_destination.to_csv(path+"midstream_CI_weighted_by_destination.csv", index=False)

# Quick peek
print(by_source.head(10))
print(by_destination.head(10))


           source  distance_km_weighted  midstream_EF_g_MJ_weighted  \
17          Texas           1158.552832                    4.638447   
16   Pennsylvania            630.653870                    2.524921   
9       Louisiana           1120.927341                    4.487808   
6   International           2478.890293                    9.924625   
15       Oklahoma            732.599464                    2.933076   
20  West Virginia            717.635732                    2.873167   
14           Ohio            874.365318                    3.500658   
12     New Mexico           1070.866041                    4.287380   
4        Colorado            676.455804                    2.708297   
21        Wyoming            856.949203                    3.430930   

    midstream_EF_g_MJ_low_weighted  midstream_EF_g_MJ_high_weighted  \
17                        4.112227                         5.257754   
16                        2.238475                         2.862039   
9    